# Árvore de decisão para regressão — Sleep Debt & Screen Time

Segunda etapa do trabalho. O dataset é o mesmo da etapa 1 (MLP), mas agora o problema é de **regressão**:
prever **quantas horas uma pessoa dorme** (`total_sleep_hours`) a partir do perfil e dos hábitos à noite,
com uma **árvore de decisão** (`DecisionTreeRegressor` do scikit-learn).

**Dataset:** [Sleep Debt & Screen Time: Late Night Phone Habits](https://www.kaggle.com/datasets/samartalwar/sleep-debt-and-screen-time-late-night-phone-habits) (Kaggle, Samar Talwar)

| Requisito do trabalho | Como o dataset atende |
|---|---|
| Algoritmo de regressão | árvore de decisão (`DecisionTreeRegressor`) |
| Alvo contínuo | `total_sleep_hours`, de 3,2 h a 9,8 h |
| ≥ 500 amostras | 8.500 amostras |
| Dataset pouco comum | não é nenhum dos clássicos de curso (Census, MNIST, preço de casas...) |
| Tratamento dos dados documentado | seção 4: remoção de colunas que vazam o alvo, one-hot, divisão treino/teste |

Roda no **Google Colab** ou **localmente** (Python 3.9+, `pip install pandas scikit-learn matplotlib`).

## 1. Configuração
Todos os parâmetros ajustáveis ficam nesta célula. Os comentários explicam o efeito de cada um.

In [ ]:
# ----------------------------------------------------------------------------
# DADOS
# ----------------------------------------------------------------------------
SEED = 42            # semente: fixa a divisão dos dados e o desempate entre divisões da árvore.
TEST_SIZE = 0.20     # fração reservada para TESTE (nunca vista no treino). 0.2 = 1.700 amostras.
ALVO = "total_sleep_hours"   # o que queremos prever: horas de sono (número contínuo)

# Colunas que NÃO entram como atributo. Ver a seção 4 para a justificativa de cada grupo.
IDENTIFICADOR = ["user_id"]                       # só um código, não carrega informação
DERIVADA_DO_ALVO = ["sleep_debt_category"]        # é uma faixa de horas de sono: entregaria a resposta
MEDIDAS_DA_NOITE = ["sleep_latency_min", "deep_sleep_pct", "rem_sleep_pct"]   # medidas da própria noite
CONSEQUENCIAS = ["next_day_fatigue_score", "morning_alarm_snoozes"]           # medidas na manhã seguinte
COLUNAS_REMOVIDAS = IDENTIFICADOR + DERIVADA_DO_ALVO + MEDIDAS_DA_NOITE + CONSEQUENCIAS

# ----------------------------------------------------------------------------
# ÁRVORE (hiperparâmetros testados na validação cruzada)
# ----------------------------------------------------------------------------
PROFUNDIDADES = list(range(1, 16)) + [None]   # max_depth: quantas perguntas em sequência, no máximo.
                                              # None = cresce até cada folha ficar pura (decora o treino).
MIN_POR_FOLHA = [1, 5, 10, 20, 50, 100]       # min_samples_leaf: mínimo de pessoas em cada folha.
                                              # Folhas maiores = previsões mais estáveis, menos detalhe.
CRITERIO = "squared_error"                    # cada divisão escolhe a pergunta que mais reduz o erro
                                              # quadrático (a variância do alvo dentro de cada lado).
FOLDS = 5                                     # partes da validação cruzada (só no treino)

## 2. Carregar os dados
Baixa o CSV direto da API pública do Kaggle (não precisa de login). Se preferir, coloque `bedtime_screentime_sleep_debt.csv` na mesma pasta do notebook (ou faça upload no Colab).

In [ ]:
import io, os, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message=".*encountered in matmul")

ARQUIVO = "bedtime_screentime_sleep_debt.csv"
URL = ("https://www.kaggle.com/api/v1/datasets/download/"
       "samartalwar/sleep-debt-and-screen-time-late-night-phone-habits")

if not os.path.exists(ARQUIVO):
    with urllib.request.urlopen(URL) as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extract(ARQUIVO)

df = pd.read_csv(ARQUIVO)
print(df.shape)
df.head()

## 3. Análise exploratória
O que olhar: tipos, valores ausentes, a distribuição do alvo e quais colunas andam junto com ele.

In [ ]:
df.info()
print("\nValores ausentes no dataset inteiro:", int(df.isna().sum().sum()))

In [ ]:
print(df[ALVO].describe().round(2))
plt.figure(figsize=(8, 3.5))
plt.hist(df[ALVO], bins=np.arange(3, 10.01, 0.25), color="tab:blue")
plt.axvline(df[ALVO].mean(), color="k", ls="--", label=f"média = {df[ALVO].mean():.2f} h")
plt.xlabel("horas de sono"); plt.ylabel("pessoas"); plt.legend(); plt.title("Distribuição do alvo")
plt.tight_layout(); plt.show()

In [ ]:
# Correlação de cada atributo numérico com as horas de sono.
# As mais fortes são consequências do sono (cansaço, soneca) e não causas: ver seção 4.
corr = df.select_dtypes("number").corr()[ALVO].drop(ALVO).sort_values()
corr.plot.barh(figsize=(7, 4), color=["tab:red" if v < 0 else "tab:blue" for v in corr])
plt.axvline(0, color="k", lw=0.8); plt.title("Correlação com as horas de sono"); plt.tight_layout(); plt.show()
corr.round(2)

In [ ]:
# Média de horas de sono por categoria (atributos categóricos)
for c in ["chronotype", "occupation_type", "primary_bedtime_app", "gender"]:
    print(df.groupby(c)[ALVO].agg(["mean", "count"]).round(2).sort_values("mean"), "\n")

## 4. Pré-processamento

1. **Remover colunas que não podem ser usadas.**
   - `user_id`: só um identificador.
   - `sleep_debt_category`: o alvo da etapa 1 é, na prática, uma faixa de horas de sono. Usá-la seria
     **vazamento do alvo** (*target leakage*): o modelo receberia a resposta disfarçada.
   - **Medidas da própria noite** (latência, sono profundo, REM) e **consequências na manhã seguinte**
     (cansaço, despertador adiado): são medidas *junto com* ou *depois* do sono que queremos prever. Num uso real
     (prever à noite quanto alguém vai dormir) elas ainda não existiriam. A seção 9 mostra o que acontece se
     forem mantidas.
2. **Separar treino e teste** (80/20) *antes* de qualquer transformação. Na regressão não há classes para estratificar;
   a divisão é aleatória e conferimos que o alvo tem distribuição parecida nos dois conjuntos.
3. **Categóricas → one-hot.** A árvore do scikit-learn só aceita números. Usar inteiros (0, 1, 2…) criaria uma
   ordem falsa entre, por exemplo, `TikTok` e `YouTube`.
4. **Numéricas: sem padronização.** Diferente da MLP, a árvore só compara cada atributo com um limiar
   (`cafeína ≤ 120?`). Mudar a escala muda o limiar, mas não a divisão: o resultado é idêntico com ou sem z-score.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

X = df.drop(columns=COLUNAS_REMOVIDAS + [ALVO])
y = df[ALVO].to_numpy()

colunas_categoricas = X.select_dtypes(include="object").columns.tolist()
colunas_numericas = X.select_dtypes(exclude="object").columns.tolist()
print("Categóricas:", colunas_categoricas)
print("Numéricas:  ", colunas_numericas)

X_train_df, X_test_df, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)

preproc = ColumnTransformer([
    ("num", "passthrough", colunas_numericas),                        # numéricas entram como estão
    ("cat", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas),
], sparse_threshold=0, verbose_feature_names_out=False)

X_train = preproc.fit_transform(X_train_df)   # fit SÓ no treino (aprende as categorias)
X_test = preproc.transform(X_test_df)
nomes_features = preproc.get_feature_names_out()

print(f"\nTreino: {X_train.shape}  Teste: {X_test.shape}")
print(f"{X.shape[1]} atributos originais -> {X_train.shape[1]} entradas após one-hot")
print(f"Alvo no treino: média {y_train.mean():.2f} h, desvio {y_train.std():.2f} h")
print(f"Alvo no teste:  média {y_test.mean():.2f} h, desvio {y_test.std():.2f} h")

In [ ]:
# Prova de que a escala não importa para a árvore: treina com e sem z-score e compara as previsões.
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

zs = StandardScaler().fit(X_train)
a = DecisionTreeRegressor(max_depth=6, random_state=SEED).fit(X_train, y_train).predict(X_test)
b = DecisionTreeRegressor(max_depth=6, random_state=SEED).fit(zs.transform(X_train), y_train).predict(zs.transform(X_test))
print("Maior diferença entre as previsões com e sem z-score:", np.abs(a - b).max())

## 5. Escolha dos hiperparâmetros (validação cruzada no treino)

Uma árvore sem limite cresce até decorar o treino. Os dois freios mais importantes são:

- `max_depth`: número máximo de perguntas em sequência;
- `min_samples_leaf`: número mínimo de pessoas em cada folha.

O `GridSearchCV` testa todas as combinações com validação cruzada de 5 partes **só no treino**. O teste continua guardado.
A métrica é o **R²**: 1 seria acertar tudo; 0 é o mesmo que prever sempre a média.

In [ ]:
from sklearn.model_selection import GridSearchCV

busca = GridSearchCV(
    DecisionTreeRegressor(criterion=CRITERIO, random_state=SEED),
    {"max_depth": PROFUNDIDADES, "min_samples_leaf": MIN_POR_FOLHA},
    cv=FOLDS, scoring="r2", return_train_score=True,
)
busca.fit(X_train, y_train)
res = pd.DataFrame(busca.cv_results_)
print("Melhor combinação:", busca.best_params_, f" R² médio na validação = {busca.best_score_:.4f}")
tabela = res.pivot_table(index="param_max_depth", columns="param_min_samples_leaf",
                         values="mean_test_score", dropna=False)
tabela.round(3)

In [ ]:
# Curva de complexidade: R² no treino x validação em função da profundidade (min_samples_leaf = 1).
r1 = res[res.param_min_samples_leaf == 1]
rotulos = [str(d) if d is not None else "sem\nlimite" for d in r1.param_max_depth]
plt.figure(figsize=(9, 4))
plt.plot(rotulos, r1.mean_train_score, "o-", label="treino")
plt.plot(rotulos, r1.mean_test_score, "o-", label="validação (CV)")
plt.xlabel("max_depth"); plt.ylabel("R²"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Árvore funda decora o treino: a validação piora depois da profundidade 6")
plt.tight_layout(); plt.show()

## 6. Treinar a árvore final
Usa a melhor combinação da busca, treinada de novo com todo o conjunto de treino.

In [ ]:
arvore = DecisionTreeRegressor(criterion=CRITERIO, random_state=SEED, **busca.best_params_)
arvore.fit(X_train, y_train)
print("Profundidade:", arvore.get_depth())
print("Folhas:", arvore.get_n_leaves(), "(cada folha devolve a média de horas das pessoas de treino que caíram nela)")
print("Nós no total:", arvore.tree_.node_count)

In [ ]:
from sklearn.tree import export_text, plot_tree

# Os três primeiros níveis em texto: cada linha é uma pergunta; "value" é a previsão (horas) naquele ponto.
print(export_text(arvore, feature_names=list(nomes_features), max_depth=3, decimals=2, show_weights=False))

In [ ]:
plt.figure(figsize=(16, 6))
plot_tree(arvore, feature_names=nomes_features, max_depth=2, filled=True, impurity=False,
          precision=2, rounded=True, fontsize=9)
plt.title("Topo da árvore (2 níveis)"); plt.show()

## 7. Avaliação no conjunto de teste
O teste só é usado aqui, depois de todas as escolhas feitas.

- **R²**: fração da variação das horas de sono que o modelo explica.
- **MAE**: erro absoluto médio, em horas.
- **RMSE**: raiz do erro quadrático médio, em horas. Pune mais os erros grandes.
- **Referência**: prever sempre a média de horas do treino.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def metricas(y_true, y_pred):
    return {"R²": r2_score(y_true, y_pred), "MAE (h)": mean_absolute_error(y_true, y_pred),
            "RMSE (h)": mean_squared_error(y_true, y_pred) ** 0.5}

pred_train = arvore.predict(X_train)
pred = arvore.predict(X_test)
media = np.full_like(y_test, y_train.mean())
pd.DataFrame({"árvore · treino": metricas(y_train, pred_train),
              "árvore · teste": metricas(y_test, pred),
              "sempre a média · teste": metricas(y_test, media)}).T.round(4)

In [ ]:
erro = pred - y_test
print(f"Erro médio de {mean_absolute_error(y_test, pred) * 60:.0f} minutos")
print(f"{np.mean(np.abs(erro) <= 0.5):.1%} das previsões erram por no máximo 30 min; "
      f"{np.mean(np.abs(erro) <= 1):.1%} por no máximo 1 h")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
a1.scatter(y_test, pred, s=8, alpha=0.4)
a1.plot([3, 10], [3, 10], "k--", lw=1, label="previsão perfeita")
a1.set_xlabel("horas reais"); a1.set_ylabel("horas previstas"); a1.legend()
a1.set_title("Previsto x real (teste)")
a2.hist(erro, bins=40, color="tab:orange")
a2.axvline(0, color="k", lw=1)
a2.set_xlabel("erro (previsto − real), em horas"); a2.set_title("Distribuição dos erros")
plt.tight_layout(); plt.show()
# As previsões formam faixas horizontais: a árvore só sabe devolver um valor por folha.
print("Valores distintos previstos no teste:", len(np.unique(pred)))

## 8. Importância dos atributos
A importância de um atributo é quanto ele reduziu o erro quadrático, somado em todas as divisões em que foi usado
(normalizado para somar 1).

In [ ]:
imp = pd.Series(arvore.feature_importances_, index=nomes_features).sort_values(ascending=False)
imp.head(10)[::-1].plot.barh(figsize=(7, 4)); plt.title("Importância (redução do erro)"); plt.tight_layout(); plt.show()
imp.round(4).head(12)

## 9. Discussão: e se usássemos as medidas do sono?

Na seção 4 removemos as medidas da própria noite e as consequências na manhã seguinte. A função abaixo repete
**a mesma divisão, o mesmo pré-processamento e a mesma busca de hiperparâmetros** com outros conjuntos de colunas.

In [ ]:
from sklearn.base import clone

def avaliar(remover):
    """Repete divisão, pré-processamento e busca do notebook removendo as colunas indicadas."""
    Xs = df.drop(columns=remover + [ALVO])
    cat = Xs.select_dtypes(include="object").columns.tolist()
    num = Xs.select_dtypes(exclude="object").columns.tolist()
    Xtr, Xte, ytr, yte = train_test_split(Xs, y, test_size=TEST_SIZE, random_state=SEED)
    pre = ColumnTransformer([("num", "passthrough", num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)],
                            sparse_threshold=0)
    b = clone(busca).fit(pre.fit_transform(Xtr), ytr)
    p = b.predict(pre.transform(Xte))
    return Xtr.shape[1], b.best_params_, r2_score(yte, p), mean_absolute_error(yte, p)

base = IDENTIFICADOR + DERIVADA_DO_ALVO
cenarios = {
    "Só perfil e hábitos (modelo final)": COLUNAS_REMOVIDAS,
    "+ medidas da noite": base + CONSEQUENCIAS,
    "+ consequências na manhã": base + MEDIDAS_DA_NOITE,
    "Todas as medidas de sono": base,
}
for nome, remover in cenarios.items():
    n, bp, r2, mae = avaliar(remover)
    print(f"{nome:36s} {n:2d} atributos  {str(bp):40s} R² = {r2:.4f}  MAE = {mae * 60:.0f} min")

### Interpretação

- **Hábitos e perfil explicam cerca de 80% da variação das horas de sono.** A árvore erra em média cerca de meia hora,
  contra mais de 1 h de quem chuta sempre a média.
- **Quatro perguntas fazem quase todo o trabalho:** minutos de celular antes de dormir, ser noturno ou matutino e trabalhar
  em turnos na área da saúde. Os outros atributos (app usado, brilho, gênero, idade) quase não aparecem.
- **A profundidade controla o overfitting.** Sem limite, a árvore chega a R² = 1 no treino e piora na validação.
  A profundidade 6 (63 folhas) foi o melhor equilíbrio.
- **Treino e teste têm o mesmo R² (cerca de 0,80):** com profundidade 6 a árvore não decorou o treino.
- **As medidas da noite (latência, sono profundo, REM) quase não acrescentam nada.** Já as consequências na manhã
  seguinte sobem o R² para cerca de 0,90, mas por um motivo ruim: cansaço e despertador adiado são *efeitos* de ter
  dormido pouco. Usá-las melhora o número, mas o modelo deixa de servir para prever o sono antes de ele acontecer.
- **Limitação da árvore:** a previsão é constante dentro de cada folha (faixas horizontais no gráfico previsto × real).
  Florestas aleatórias ou boosting suavizam isso combinando muitas árvores.

## 10. Comparação com a etapa 1 (MLP)

| | Etapa 1: MLP | Etapa 2: árvore de decisão |
|---|---|---|
| Tarefa | classificação (4 categorias de dívida de sono) | regressão (horas de sono) |
| Numéricas | padronização z-score (obrigatória) | sem transformação (escala não importa) |
| Categóricas | one-hot | one-hot |
| Hiperparâmetros principais | camadas, taxa de aprendizado, early stopping | `max_depth`, `min_samples_leaf` |
| Como evita overfitting | early stopping e L2 | limite de profundidade e tamanho mínimo das folhas |
| Interpretação | difícil (1.556 pesos) | cada previsão é uma sequência de perguntas legíveis |